# Week 2 — Gaussian Process optimisation

Update the models using the observations available at the start of Week 2 and select the next query for each function.


In [13]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern
from sklearn.exceptions import ConvergenceWarning


working_dir = Path.cwd()
possible_data_folders = [working_dir.parent / "Data", working_dir / "Data"]
data_folder = next((p for p in possible_data_folders if all((p / f"function_{i}.csv").is_file() for i in range(1, 9))), None)
if data_folder is None:
    raise FileNotFoundError(f"Cannot find function_1.csv through function_8.csv in a Data folder near {working_dir}")
print("Loading Week 2 observations from:", data_folder)


Loading Week 2 observations from: C:\Users\charl\OneDrive - thesuperyachtgroup.com\Desktop\Imperial AI course\7. Capstone Project\2. Module 13\Data


## 1. Load the cumulative Week 2 observations


In [14]:
data = {}
for i in range(1, 9):
    df = pd.read_csv(data_folder / f"function_{i}.csv")
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed:")]
    df.columns = df.columns.str.strip()
    if "output" not in df.columns:
        raise ValueError(f"function_{i}.csv needs an 'output' column; found {list(df.columns)}")
    input_cols = [c for c in df.columns if c != "output"]
    expected = [f"x{j+1}" for j in range(len(input_cols))]
    if set(input_cols) != set(expected):
        raise ValueError(f"function_{i}.csv has unexpected input columns: {input_cols}")
    df = df[expected + ["output"]].apply(pd.to_numeric, errors="raise")
    data[i] = df
    print(f"Function {i}: {len(df)} observations, {len(expected)} dimensions")


Function 1: 11 observations, 2 dimensions
Function 2: 11 observations, 2 dimensions
Function 3: 16 observations, 3 dimensions
Function 4: 31 observations, 4 dimensions
Function 5: 21 observations, 4 dimensions
Function 6: 21 observations, 5 dimensions
Function 7: 31 observations, 6 dimensions
Function 8: 41 observations, 8 dimensions


## 2. Compare the returned observations with previous best outputs


In [15]:
for i in range(1, 9):
    df = data[i]
    if len(df) < 2:
        raise ValueError(f"Function {i} needs at least two observations")
    new_output = df.iloc[-1]["output"]
    previous_best = df.iloc[:-1]["output"].max()
    updated_best = df["output"].max()
    print(f"Function {i}: previous best={previous_best:.6f}, new={new_output:.6f}, "
          f"difference={new_output-previous_best:+.6f}, updated best={updated_best:.6f}, "
          f"new best={bool(new_output > previous_best)}")


Function 1: previous best=0.000000, new=0.000000, difference=-0.000000, updated best=0.000000, new best=False
Function 2: previous best=0.611205, new=-0.082041, difference=-0.693246, updated best=0.611205, new best=False
Function 3: previous best=-0.034835, new=-0.020167, difference=+0.014668, updated best=-0.020167, new best=True
Function 4: previous best=-4.025542, new=-0.300645, difference=+3.724898, updated best=-0.300645, new best=True
Function 5: previous best=1088.859618, new=2329.015854, difference=+1240.156236, updated best=2329.015854, new best=True
Function 6: previous best=-0.714265, new=-2.661556, difference=-1.947291, updated best=-0.714265, new best=False
Function 7: previous best=1.364968, new=1.950612, difference=+0.585644, updated best=1.950612, new best=True
Function 8: previous best=9.598482, new=9.821986, difference=+0.223504, updated best=9.821986, new best=True


## 3. Refit the RBF Gaussian Processes


In [16]:
models_week2 = {}
fit_rows = []
for i in range(1, 9):
    df = data[i]
    X = df.drop(columns="output").values
    y = df["output"].values
    kernel = RBF(length_scale=0.5, length_scale_bounds=(1e-5, 1e5))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=10, random_state=42)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        gp.fit(X, y)
    models_week2[i] = gp
    fit_rows.append({"function": i, "fitted kernel": str(gp.kernel_),
                     "convergence warning": any(issubclass(w.category, ConvergenceWarning) for w in caught)})
print(pd.DataFrame(fit_rows).to_string(index=False))


 function            fitted kernel  convergence warning
        1  RBF(length_scale=1e-05)                 True
        2 RBF(length_scale=0.0453)                False
        3  RBF(length_scale=0.202)                False
        4  RBF(length_scale=0.522)                False
        5  RBF(length_scale=0.225)                False
        6  RBF(length_scale=0.481)                False
        7  RBF(length_scale=0.354)                False
        8  RBF(length_scale=0.913)                False


## 4. Compare RBF and Matérn for Function 1


In [17]:
from sklearn.gaussian_process.kernels import Matern

df = data[1]

X = df.drop(columns="output").values
y = df["output"].values

rbf_kernel = RBF(
    length_scale=0.5,
    length_scale_bounds=(1e-5, 1e5)
)

rbf_gp = GaussianProcessRegressor(
    kernel=rbf_kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

rbf_gp.fit(X, y)


matern_kernel = Matern(
    length_scale=0.5,
    length_scale_bounds=(1e-5, 1e5),
    nu=2.5
)

matern_gp = GaussianProcessRegressor(
    kernel=matern_kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

matern_gp.fit(X, y)


print("RBF kernel:")
print(rbf_gp.kernel_)

print("\nMatérn kernel:")
print(matern_gp.kernel_)


RBF kernel:
RBF(length_scale=1e-05)

Matérn kernel:
Matern(length_scale=1e-05, nu=2.5)


## 5. Compare the predictions for Function 1


In [18]:
rng = np.random.default_rng(42)

X_candidates_f1 = rng.uniform(
    low=0,
    high=1,
    size=(100000, 2)
)

rbf_mean, rbf_std = rbf_gp.predict(
    X_candidates_f1,
    return_std=True
)

matern_mean, matern_std = matern_gp.predict(
    X_candidates_f1,
    return_std=True
)

print("=" * 60)
print("FUNCTION 1 - RBF vs MATERN")
print("=" * 60)

print("\nRBF")
print(f"Predicted output range: {rbf_mean.min():.10f} to {rbf_mean.max():.10f}")
print(f"Uncertainty range:      {rbf_std.min():.10f} to {rbf_std.max():.10f}")

print("\nMATERN")
print(f"Predicted output range: {matern_mean.min():.10f} to {matern_mean.max():.10f}")
print(f"Uncertainty range:      {matern_std.min():.10f} to {matern_std.max():.10f}")


FUNCTION 1 - RBF vs MATERN

RBF
Predicted output range: -0.0003278239 to -0.0000001931
Uncertainty range:      0.0000355799 to 0.0010366701

MATERN
Predicted output range: -0.0003278239 to -0.0000003215
Uncertainty range:      0.0000459032 to 0.0010366701


## 6. Generate candidate points and apply UCB


In [19]:
week2_candidates = {}
ucb_rows = []
rng = np.random.default_rng(42)
kappa = 2.5
for i in range(2, 9):
    gp = models_week2[i]
    n_dimensions = data[i].shape[1] - 1
    X_candidates = rng.uniform(0, 1, size=(100000, n_dimensions))
    mean, std = gp.predict(X_candidates, return_std=True)
    ucb = mean + kappa * std
    best_index = int(np.argmax(ucb))
    selected_point = X_candidates[best_index]
    week2_candidates[i] = selected_point
    ucb_rows.append({"function": i, "predicted mean": mean[best_index],
                     "uncertainty": std[best_index], "UCB": ucb[best_index],
                     "query coordinates": " ".join(f"{v:.6f}" for v in selected_point)})
print(f"UCB candidate selection (kappa={kappa})")
print(pd.DataFrame(ucb_rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))


UCB candidate selection (kappa=2.5)
 function  predicted mean  uncertainty         UCB                                                       query coordinates
        2        0.441071     0.190032    0.916151                                                       0.655286 0.929896
        3       -0.055801     0.074303    0.129955                                              0.397530 0.090477 0.447319
        4       -1.666506     0.886853    0.550627                                     0.480756 0.474000 0.229059 0.429178
        5     2112.903796   303.291367 2871.132212                                     0.514842 0.856105 0.961506 0.998492
        6       -0.664546     0.355483    0.224161                            0.246549 0.344645 0.479525 0.733976 0.025332
        7        1.772966     0.240968    2.375386                   0.080334 0.317465 0.485517 0.055569 0.406883 0.931976
        8        9.712587     0.462513   10.868871 0.078127 0.668711 0.054188 0.209488 0.863552 0.71779

## 7. Compare proposed queries with current best observations


In [20]:
distance_rows = []
for i in range(2, 9):
    df = data[i]
    best_row = df.loc[df["output"].idxmax()]
    best_x = best_row.drop("output").to_numpy(dtype=float)
    distance_rows.append({"function": i, "current best output": best_row["output"],
                          "distance from best": np.linalg.norm(week2_candidates[i] - best_x)})
print("Proposed UCB queries relative to current best observations")
print(pd.DataFrame(distance_rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Proposed UCB queries relative to current best observations
 function  current best output  distance from best
        2             0.611205            0.047468
        3            -0.020167            0.317077
        4            -0.300645            0.160930
        5          2329.015854            0.165759
        6            -0.714265            0.578483
        7             1.950612            0.249013
        8             9.821986            0.918984


## 8. Select an exploration point for Function 1


In [21]:
from scipy.spatial.distance import cdist

X_existing = data[1].drop(columns="output").values

X_candidates = X_candidates_f1

distances = cdist(X_candidates, X_existing)

nearest_distance = distances.min(axis=1)

best_index = np.argmax(nearest_distance)

function1_point = X_candidates[best_index]
function1_distance = nearest_distance[best_index]

print("=" * 60)
print("FUNCTION 1 - EXPLORATION")
print("=" * 60)

print("Selected point:")
print(function1_point)

print("\nDistance to nearest existing observation:")
print(function1_distance)


FUNCTION 1 - EXPLORATION
Selected point:
[0.99997255 0.99984253]

Distance to nearest existing observation:
0.3456413874867929


## 9. Combine the final query points


In [22]:

final_points = {
    1: function1_point,
    2: week2_candidates[2],
    3: week2_candidates[3],
    4: week2_candidates[4],
    5: week2_candidates[5],
    6: week2_candidates[6],
    7: week2_candidates[7],
    8: week2_candidates[8]
}


## 10. Validate the final queries


In [23]:
validation_rows = []
for i in range(1, 9):
    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values
    min_distance = np.linalg.norm(X_existing - point, axis=1).min()
    duplicate = bool(np.any(np.all(np.isclose(X_existing, point), axis=1)))
    in_bounds = bool(np.all((point >= 0) & (point <= 1)))
    method = "max-distance exploration" if i == 1 else "GP + UCB"
    verified = bool(np.allclose(point, X_candidates_f1[np.argmax(nearest_distance)])) if i == 1 else bool(np.allclose(point, week2_candidates[i]))
    validation_rows.append({"function": i, "method": method, "in bounds": in_bounds,
                            "duplicate": duplicate, "nearest distance": min_distance,
                            "selection matches": verified})
print("Final query validation")
print(pd.DataFrame(validation_rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))
assert all(row["in bounds"] and not row["duplicate"] and row["selection matches"] for row in validation_rows)


Final query validation
 function                   method  in bounds  duplicate  nearest distance  selection matches
        1 max-distance exploration       True      False          0.345641               True
        2                 GP + UCB       True      False          0.047468               True
        3                 GP + UCB       True      False          0.291689               True
        4                 GP + UCB       True      False          0.160930               True
        5                 GP + UCB       True      False          0.165759               True
        6                 GP + UCB       True      False          0.562812               True
        7                 GP + UCB       True      False          0.249013               True
        8                 GP + UCB       True      False          0.774658               True


## 11. Display the final query points


In [24]:
queries = pd.DataFrame([
    {"function": i, "method": "max-distance exploration" if i == 1 else "GP + UCB",
     "proposed query coordinates": " ".join(f"{v:.8f}" for v in final_points[i])}
    for i in range(1, 9)
])
print("Week 2 — final proposed queries")
print(queries.to_string(index=False))


Week 2 — final proposed queries
 function                   method                                                              proposed query coordinates
        1 max-distance exploration                                                                   0.99997255 0.99984253
        2                 GP + UCB                                                                   0.65528565 0.92989635
        3                 GP + UCB                                                        0.39752993 0.09047714 0.44731895
        4                 GP + UCB                                             0.48075601 0.47400033 0.22905919 0.42917843
        5                 GP + UCB                                             0.51484244 0.85610509 0.96150603 0.99849219
        6                 GP + UCB                                  0.24654859 0.34464495 0.47952502 0.73397600 0.02533225
        7                 GP + UCB                       0.08033361 0.31746516 0.48551741 0.05556860 0.4068